# Yellow Taxi Data Analysis for Whole 2023 Year

In [1]:
# Try use the data in 20023
import polars as pl
import glob
import os
import gc
import glob

## Step 1: Load Lazy Mode Through Polars Scan

In [2]:
# C:\Users\ekadw\Documents\DATA\NY_Taxi\2009\yellow_taxi
files = glob.glob(r"C:\Users\ekadw\Documents\DATA\NY_Taxi\2023\yellow_taxi\yellow_tripdata_2023-*.parquet", recursive=True)
yellow_2023 = pl.scan_parquet(files)

## Step 2: Features Engineering

In [3]:
mapping = {
    "Credit": 0,
    "CREDIT": 0,
    "CASH": 1,
    "Cash": 1,
    "No Charge": 2,
    "Dispute": 3
}

In [ ]:
yellow_2023 = (
    yellow_2023.select(["Trip_Pickup_DateTime", "Trip_Dropoff_DateTime", "Passenger_Count", "Trip_Distance", "Payment_Type",
                        "Fare_Amt", "Tip_Amt"])
               .filter((pl.col("Passenger_Count") >= 0) & (pl.col("Trip_Distance") >= 0) & (pl.col("Trip_Distance") <= 50) & 
                       (pl.col("Fare_Amt") >= 0) & (pl.col("Tip_Amt") >= 0))
               .with_columns(
                       pl.col("Payment_Type").replace(mapping))
               .with_columns(
                       pl.col("Payment_Type").cast(pl.Int64))
               .filter(
                       pl.col("Payment_Type") == 0)
               .with_columns([
                       pl.col("Trip_Pickup_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Trip_Dropoff_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Payment_Type").cast(pl.Int64)])
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds() / 86400)
                               .cast(pl.Int64)
                               .alias("Duration_Days"))
               .filter(
                       (pl.col("Duration_Days") == 0))
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds())
                               .cast(pl.Int64)
                               .alias("Duration_Seconds"))
               .with_columns(
                       pl.when(pl.col("Tip_Amt") <= 0.0).then(1)
                               .otherwise(0)
                               .alias("Tip_Category"))
               .select(["Passenger_Count", "Trip_Distance", "Fare_Amt", "Duration_Seconds", "Tip_Category"])
)

In [ ]:
import polars as pl
import numpy as np
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc
import matplotlib.pyplot as plt
import glob

# Initialize model (logistic regression using SGD)
clf = SGDClassifier(loss="log_loss")
classes = [0,1]  # adjust to your Payment_Type labels

# For storing true labels and predicted probabilities (for ROC/metrics)
y_true_all = []
y_pred_all = []
y_score_all = []

batch_size = 100_000
files = glob.glob(r"C:\Users\ekadw\Documents\DATA\NY_Taxi\2023\yellow_taxi\yellow_tripdata_2023-*.parquet")

for f in files:
    # Build lazy frame (data manipulation happens here)
    lf = (
        pl.scan_parquet(f)
        .select(["Trip_Pickup_DateTime", "Trip_Dropoff_DateTime", "Passenger_Count", "Trip_Distance", "Payment_Type",
                        "Fare_Amt", "Tip_Amt"])
               .filter((pl.col("Passenger_Count") >= 0) & (pl.col("Trip_Distance") >= 0) & (pl.col("Trip_Distance") <= 50) & 
                       (pl.col("Fare_Amt") >= 0) & (pl.col("Tip_Amt") >= 0))
               .with_columns(
                       pl.col("Payment_Type").replace(mapping))
               .with_columns(
                       pl.col("Payment_Type").cast(pl.Int64))
               .filter(
                       pl.col("Payment_Type") == 0)
               .with_columns([
                       pl.col("Trip_Pickup_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Trip_Dropoff_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Payment_Type").cast(pl.Int64)])
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds() / 86400)
                               .cast(pl.Int64)
                               .alias("Duration_Days"))
               .filter(
                       (pl.col("Duration_Days") == 0))
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds())
                               .cast(pl.Int64)
                               .alias("Duration_Seconds"))
               .with_columns(
                       pl.when(pl.col("Tip_Amt") <= 0.0).then(1)
                               .otherwise(0)
                               .alias("Tip_Category"))
               .select(["Passenger_Count", "Trip_Distance", "Fare_Amt", "Duration_Seconds", "Tip_Category"])
    )

    # Stream through this file in batches
    start = 0
    while True:
        # Collect only a tiny slice to keep memory low
        batch = lf.slice(start, batch_size).collect(streaming=True)
        if batch.height == 0:
            break  # no more rows

        # Prepare features/labels
        X = batch.drop("Tip_Category").to_numpy()
        y = batch["Tip_Category"].to_numpy()

        # Train incrementally
        clf.partial_fit(X, y, classes=classes)

        # Store predictions for metrics
        y_pred = clf.predict(X)
        y_score = clf.predict_proba(X)  # needed for ROC

        y_true_all.extend(y)
        y_pred_all.extend(y_pred)
        y_score_all.extend(y_score)

        start += batch_size

# --- Calculate metrics ---
y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)
y_score_all = np.array(y_score_all)

accuracy = accuracy_score(y_true_all, y_pred_all)
precision = precision_score(y_true_all, y_pred_all, average='macro')
recall = recall_score(y_true_all, y_pred_all, average='macro')
f1 = f1_score(y_true_all, y_pred_all, average='macro')

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

# --- ROC curve (macro average for multi-class) ---
# sklearn's roc_curve works for one-vs-rest binary targets,
# so we need to binarize labels for multi-class ROC.
from sklearn.preprocessing import label_binarize
n_classes = len(classes)
y_true_bin = label_binarize(y_true_all, classes=classes)

# Calculate ROC curve and AUC for each class
fpr, tpr, roc_auc = dict(), dict(), dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score_all[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curves
plt.figure(figsize=(8,6))
for i in range(n_classes):
    plt.plot(fpr[i], tpr[i], label=f"Class {classes[i]} (AUC = {roc_auc[i]:.2f})")

plt.plot([0, 1], [0, 1], 'k--', label='Chance')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (One-vs-Rest)")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


In [ ]:
from sklearn.linear_model import SGDClassifier

# Initialize model
clf = SGDClassifier(loss="log_loss")
classes = [0, 1]  # adjust to your labels

# Loop through files
#files = glob.glob("yellow_2009/*.parquet")
for f in files:
    lf = (
        pl.scan_parquet(f)
        .select(["Trip_Pickup_DateTime", "Trip_Dropoff_DateTime", "Passenger_Count", "Trip_Distance", "Payment_Type",
                        "Fare_Amt", "Tip_Amt"])
               .filter((pl.col("Passenger_Count") >= 0) & (pl.col("Trip_Distance") >= 0) & (pl.col("Trip_Distance") <= 50) & 
                       (pl.col("Fare_Amt") >= 0) & (pl.col("Tip_Amt") >= 0))
               .with_columns(
                       pl.col("Payment_Type").replace(mapping))
               .with_columns(
                       pl.col("Payment_Type").cast(pl.Int64))
               .filter(
                       pl.col("Payment_Type") == 0)
               .with_columns([
                       pl.col("Trip_Pickup_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Trip_Dropoff_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Payment_Type").cast(pl.Int64)])
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds() / 86400)
                               .cast(pl.Int64)
                               .alias("Duration_Days"))
               .filter(
                       (pl.col("Duration_Days") == 0))
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds())
                               .cast(pl.Int64)
                               .alias("Duration_Seconds"))
               .with_columns(
                       pl.when(pl.col("Tip_Amt") <= 0.0).then(1)
                               .otherwise(0)
                               .alias("Tip_Category"))
               .select(["Passenger_Count", "Trip_Distance", "Fare_Amt", "Duration_Seconds", "Tip_Category"])
    )

    # Collect file in small pieces
    start = 0
    batch_size = 100_000  # start small and increase only if RAM allows
    while True:
        df = lf.slice(start, batch_size).collect()
        if df.height == 0:
            break

        X = df.drop("Tip_Category").to_numpy()
        y = df["Tip_Category"].to_numpy()

        clf.partial_fit(X, y, classes=classes)
        start += batch_size

    # --- Evaluate after training ---
    y_true_list = []
    y_score_list = []  # will store probabilities for ROC

    while True:
        df = lf.slice(start, batch_size).collect()
        if df.height == 0:
            break

        X = df.drop("Tip_Category").to_numpy()
        y = df["Tip_Category"].to_numpy()

        clf.partial_fit(X, y, classes=classes)
        start += batch_size


# Combine all batches
y_true = np.concatenate(y_true_list)
y_score = np.concatenate(y_score_list)

# --- Classification metrics ---
y_pred = (y_score >= 0.5).astype(int)  # hard labels for metrics

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="binary")
rec = recall_score(y_true, y_pred, average="binary")
f1 = f1_score(y_true, y_pred, average="binary")

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)


In [5]:
import polars as pl
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, roc_auc_score
import numpy as np
import matplotlib.pyplot as plt

clf = SGDClassifier(loss="log_loss")

batch_size = 100000

# --- Train model in batches ---
for file in files:
    lf = pl.scan_parquet(file)
    start = 0
    while True:
        df = lf.(select(["Trip_Pickup_DateTime", "Trip_Dropoff_DateTime", "Passenger_Count", "Trip_Distance", "Payment_Type",
                        "Fare_Amt", "Tip_Amt"])
               .filter((pl.col("Passenger_Count") >= 0) & (pl.col("Trip_Distance") >= 0) & (pl.col("Trip_Distance") <= 50) & 
                       (pl.col("Fare_Amt") >= 0) & (pl.col("Tip_Amt") >= 0))
               .with_columns(
                       pl.col("Payment_Type").replace(mapping))
               .with_columns(
                       pl.col("Payment_Type").cast(pl.Int64))
               .filter(
                       pl.col("Payment_Type") == 0)
               .with_columns([
                       pl.col("Trip_Pickup_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Trip_Dropoff_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Payment_Type").cast(pl.Int64)])
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds() / 86400)
                               .cast(pl.Int64)
                               .alias("Duration_Days"))
               .filter(
                       (pl.col("Duration_Days") == 0))
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds())
                               .cast(pl.Int64)
                               .alias("Duration_Seconds"))
               .with_columns(
                       pl.when(pl.col("Tip_Amt") <= 0.0).then(1)
                               .otherwise(0)
                               .alias("Tip_Category"))
               .select(["Passenger_Count", "Trip_Distance", "Fare_Amt", "Duration_Seconds", "Tip_Category"])).slice(start, batch_size).collect(streaming=True)
        if df.is_empty():
            break

        del df
        gc.collect()

        y = df["Tip_Category"].to_numpy()
        X = df.drop("Tip_Category").to_numpy()

        if start == 0:
            clf.partial_fit(X, y, classes=np.unique(y))
        else:
            clf.partial_fit(X, y)
    
        start += batch_size

# --- Evaluate after training ---
y_true_list = []
y_score_list = []  # will store probabilities for ROC

for file in files:
    lf = pl.scan_parquet(file)
    start = 0
    while True:
        df = lf.(select(["Trip_Pickup_DateTime", "Trip_Dropoff_DateTime", "Passenger_Count", "Trip_Distance", "Payment_Type",
                        "Fare_Amt", "Tip_Amt"])
               .filter((pl.col("Passenger_Count") >= 0) & (pl.col("Trip_Distance") >= 0) & (pl.col("Trip_Distance") <= 50) & 
                       (pl.col("Fare_Amt") >= 0) & (pl.col("Tip_Amt") >= 0))
               .with_columns(
                       pl.col("Payment_Type").replace(mapping))
               .with_columns(
                       pl.col("Payment_Type").cast(pl.Int64))
               .filter(
                       pl.col("Payment_Type") == 0)
               .with_columns([
                       pl.col("Trip_Pickup_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Trip_Dropoff_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Payment_Type").cast(pl.Int64)])
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds() / 86400)
                               .cast(pl.Int64)
                               .alias("Duration_Days"))
               .filter(
                       (pl.col("Duration_Days") == 0))
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds())
                               .cast(pl.Int64)
                               .alias("Duration_Seconds"))
               .with_columns(
                       pl.when(pl.col("Tip_Amt") <= 0.0).then(1)
                               .otherwise(0)
                               .alias("Tip_Category"))
               .select(["Passenger_Count", "Trip_Distance", "Fare_Amt", "Duration_Seconds", "Tip_Category"])).slice(start, batch_size).collect(streaming=True)
        if df.is_empty():
            break

        del df
        gc.collect()

        y = df["Tip_Category"].to_numpy()
        X = df.drop("Tip_Category").to_numpy()
    
        proba = clf.predict_proba(X)        # get probabilities
        y_score = proba[:, 1]               # use class 1 prob if binary
    
        y_true_list.append(y)
        y_score_list.append(y_score)
    
        start += batch_size

# Combine all batches
y_true = np.concatenate(y_true_list)
y_score = np.concatenate(y_score_list)

# --- Classification metrics ---
y_pred = (y_score >= 0.5).astype(int)  # hard labels for metrics

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="binary")
rec = recall_score(y_true, y_pred, average="binary")
f1 = f1_score(y_true, y_pred, average="binary")

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1 Score :", f1)

SyntaxError: invalid syntax (153098937.py, line 16)

In [ ]:
# --- ROC curve plot ---
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = roc_auc_score(y_true, y_score)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="blue", label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC)")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
''' data manipulation
yellow_2009 = (
    yellow_2009.select(["Trip_Pickup_DateTime", "Trip_Dropoff_DateTime", "Passenger_Count", "Trip_Distance", "Payment_Type",
                        "Fare_Amt", "Tip_Amt"])
               .filter((pl.col("Passenger_Count") >= 0) & (pl.col("Trip_Distance") >= 0) & (pl.col("Trip_Distance") <= 50) & 
                       (pl.col("Fare_Amt") >= 0) & (pl.col("Tip_Amt") >= 0))
               .with_columns(
                       pl.col("Payment_Type").replace(mapping))
               .with_columns(
                       pl.col("Payment_Type").cast(pl.Int64))
               .filter(
                       pl.col("Payment_Type") == 0)
               .with_columns([
                       pl.col("Trip_Pickup_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Trip_Dropoff_DateTime").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S"),
                       pl.col("Payment_Type").cast(pl.Int64)])
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds() / 86400)
                               .cast(pl.Int64)
                               .alias("Duration_Days"))
               .filter(
                       (pl.col("Duration_Days") == 0))
               .with_columns(
                       ((pl.col("Trip_Dropoff_DateTime") - pl.col("Trip_Pickup_DateTime")).dt.total_seconds())
                               .cast(pl.Int64)
                               .alias("Duration_Seconds"))
               .with_columns(
                       pl.when(pl.col("Tip_Amt") <= 0.0).then(1)
                               .otherwise(0)
                               .alias("Tip_Category"))
               .select(["Passenger_Count", "Trip_Distance", "Fare_Amt", "Duration_Seconds", "Tip_Category"])
)
''' 